# Actividad 4 — Benchmarking de Modelos y Frameworks Agénticos
_____

_Proyecto Integrador MNA 10_

**Equipo 16**:
- Joel
- Roberto
- David

**Fecha**: 31 de Mayo, 2026


# Objetivos

* 4.1 Evaluar el rendimiento de los modelos de Amazon Bedrock mediante métricas de latencia, uso de tokens y calidad de respuesta.
* 4.2 Comparar el comportamiento de los modelos con y sin Extended Thinking habilitado.
* 4.3 Medir el rendimiento end-to-end de los frameworks agénticos (LangChain, Strands Agents, LangGraph) desplegados con Amazon Bedrock AgentCore.

## Contexto

En el Avance 3 validamos el baseline de retrieval con **S3 Vectors + Titan V2** y establecimos métricas de calidad (MRR, Precision@5, score coseno). Con el vector store productivo en su lugar, el siguiente paso es seleccionar y configurar el modelo generativo que consumirá el contexto recuperado para producir hallazgos de auditoría.

Este avance responde tres preguntas clave:

1. **¿Qué modelo de Bedrock ofrece el mejor balance latencia/calidad para nuestro caso de uso?**
2. **¿El Extended Thinking mejora la calidad de las respuestas de auditoría lo suficiente para justificar su costo adicional en latencia y tokens?**
3. **¿Qué framework agéntico (LangChain, Strands Agents, LangGraph) introduce menor overhead al orquestar el pipeline RAG completo?**

### Modelos evaluados

| Proveedor | Modelo | Model ID | Tipo |
|---|---|---|---|
| Anthropic | Claude Sonnet 4 | `us.anthropic.claude-sonnet-4-20250514-v1:0` | Propietario |
| Anthropic | Claude Haiku 3.5 | `us.anthropic.claude-3-5-haiku-20241022-v1:0` | Propietario |
| OpenAI | GPT OSS 120B | `openai.gpt-oss-120b-v1:0` | Open-weight |
| OpenAI | GPT OSS 20B | `openai.gpt-oss-20b-v1:0` | Open-weight |
| Mistral AI | Mistral Large 3 (675B) | `mistral.mistral-large-3-v1:0` | Open-weight |
| Meta | Llama 4 Maverick 17B | `us.meta.llama4-maverick-17b-instruct-v1:0` | Open-weight |
| Meta | Llama 4 Scout 17B | `us.meta.llama4-scout-17b-instruct-v1:0` | Open-weight |
| Qwen | Qwen3 32B | `qwen.qwen3-32b-v1:0` | Open-weight |
| DeepSeek | DeepSeek-R1 | `us.deepseek.r1-v1:0` | Open-weight |
| Amazon | Nova Pro | `us.amazon.nova-pro-v1:0` | Propietario |
| Amazon | Nova Lite | `us.amazon.nova-lite-v1:0` | Propietario |

### Métricas evaluadas

| Categoría | Métrica | Descripción |
|---|---|---|
| Latencia | Time to First Token (TTFT) | Tiempo hasta recibir el primer token de respuesta |
| Latencia | Invocation Latency | Tiempo total de la invocación (request → último token) |
| Tokens | Input Tokens | Tokens consumidos en el prompt (incluye contexto RAG) |
| Tokens | Output Tokens | Tokens generados en la respuesta |
| Tokens | Thinking Tokens | Tokens usados en razonamiento interno (Extended Thinking) |
| Respuesta | Response Length | Longitud en caracteres de la respuesta final |
| Configuración | Temperature, Top-P, Max Tokens | Parámetros del modelo |
| Framework | Agent Overhead | Latencia adicional introducida por el framework de orquestación |
| Framework | Total Cycles | Número de ciclos del agent loop |


## 1. Configuración del Entorno

Configuramos los clientes de AWS (Bedrock Runtime para generación, S3 Vectors para retrieval) y definimos los modelos a evaluar. Usamos la API **Converse** de Bedrock que proporciona métricas de latencia y tokens de forma nativa en la respuesta.


In [ ]:
import json
import time
import statistics
import urllib3
from datetime import datetime
from pathlib import Path

import boto3
import pandas as pd
import numpy as np

# Suppress SSL warnings (local network config)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# AWS Configuration
AWS_REGION = "us-west-2"
AWS_PROFILE = "Developer"

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock = session.client("bedrock-runtime", verify=False)
s3vectors = session.client("s3vectors", verify=False)

# Vector store constants (from Avance 3)
VECTOR_BUCKET_NAME = "iso27001-vectors-e16"
INDEX_NAME = "iso27001-compliance-index"
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"

# Models to benchmark
MODELS = {
    # --- Anthropic ---
    "claude-sonnet-4": "us.anthropic.claude-sonnet-4-20250514-v1:0",
    "claude-haiku-3.5": "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    # --- OpenAI (GPT OSS) ---
    "gpt-oss-120b": "openai.gpt-oss-120b-v1:0",
    "gpt-oss-20b": "openai.gpt-oss-20b-v1:0",
    # --- Mistral AI ---
    "mistral-large-3": "mistral.mistral-large-3-v1:0",
    # --- Meta (Llama 4) ---
    "llama4-maverick-17b": "us.meta.llama4-maverick-17b-instruct-v1:0",
    "llama4-scout-17b": "us.meta.llama4-scout-17b-instruct-v1:0",
    # --- Qwen ---
    "qwen3-32b": "qwen.qwen3-32b-v1:0",
    # --- DeepSeek ---
    "deepseek-r1": "us.deepseek.r1-v1:0",
    # --- Amazon ---
    "nova-pro": "us.amazon.nova-pro-v1:0",
    "nova-lite": "us.amazon.nova-lite-v1:0",
}

# Models that support Extended Thinking / Reasoning
THINKING_MODELS = {
    "claude-sonnet-4-thinking": "us.anthropic.claude-sonnet-4-20250514-v1:0",
    "deepseek-r1-reasoning": "us.deepseek.r1-v1:0",
    "qwen3-32b-thinking": "qwen.qwen3-32b-v1:0",
}

print(f"Region: {AWS_REGION}")
print(f"Models to benchmark: {len(MODELS)}")
print(f"Thinking/Reasoning models: {len(THINKING_MODELS)}")
print(f"Timestamp: {datetime.now().isoformat()}")
print("\nModelos incluidos:")
for name, mid in MODELS.items():
    print(f"  {name:.<25} {mid}")


## 2. Queries de Evaluación

Definimos un conjunto de queries representativas del dominio ISO 27001 que ejercitan diferentes capacidades del modelo: análisis de brechas, generación de hallazgos, y recomendaciones de remediación. Cada query se ejecutará con contexto RAG recuperado de S3 Vectors.


In [ ]:
# Evaluation queries - representative of the multi-agent audit system
EVAL_QUERIES = [
    {
        "id": "gap_analysis",
        "query": "Analiza la política de control de acceso del patrocinador y determina si cumple con el control A.9.1.1 de ISO 27001:2013. Identifica brechas específicas.",
        "type": "analysis",
    },
    {
        "id": "finding_generation",
        "query": "Genera un hallazgo de no conformidad menor para una organización que no documenta las revisiones periódicas de derechos de acceso. Usa el formato estándar de auditoría ISO.",
        "type": "generation",
    },
    {
        "id": "remediation",
        "query": "Proporciona un plan de remediación para una no conformidad mayor en gestión de incidentes de seguridad (A.16.1). Incluye plazos y responsables sugeridos.",
        "type": "recommendation",
    },
    {
        "id": "risk_assessment",
        "query": "Evalúa el nivel de riesgo asociado a la falta de un programa formal de concientización en seguridad de la información según ISO 27001 cláusula 7.3.",
        "type": "assessment",
    },
    {
        "id": "control_mapping",
        "query": "Mapea los controles de la política de respaldos del patrocinador contra los requisitos del Anexo A.12.3 (Respaldo de información). Indica cumplimiento parcial o total.",
        "type": "mapping",
    },
]

print(f"Queries de evaluación: {len(EVAL_QUERIES)}")
for q in EVAL_QUERIES:
    print(f"  [{q['id']}] ({q['type']}) {q['query'][:60]}...")


## 3. Recuperación de Contexto RAG

Reutilizamos la función de búsqueda semántica de S3 Vectors del Avance 3 para recuperar chunks relevantes que servirán como contexto para cada query. Esto simula el pipeline RAG completo que usará el sistema multi-agente.


In [ ]:
def get_embedding(text: str) -> list[float]:
    """Generate embedding using Titan V2."""
    body = json.dumps({"inputText": text, "dimensions": 1024, "normalize": True})
    response = bedrock.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=body,
        contentType="application/json",
        accept="application/json",
    )
    return json.loads(response["body"].read())["embedding"]


def search_s3vectors(query: str, top_k: int = 3) -> list[dict]:
    """Search S3 Vectors index and return ranked results."""
    query_vector = get_embedding(query)
    response = s3vectors.query_vectors(
        vectorBucketName=VECTOR_BUCKET_NAME,
        indexName=INDEX_NAME,
        queryVector={"dataFloat32": query_vector},
        topK=top_k,
        returnMetadata=True,
        returnDistance=True,
    )
    results = []
    for i, vec in enumerate(response.get("vectors", [])):
        metadata = {m["key"]: m["value"] for m in vec.get("metadata", {}).get("filterable", [])}
        non_filterable = {m["key"]: m["value"] for m in vec.get("metadata", {}).get("nonFilterable", [])}
        results.append({
            "rank": i + 1,
            "score": round(1 - vec.get("distance", 1), 4),
            "source": metadata.get("source", "unknown"),
            "folder": metadata.get("folder", "unknown"),
            "text": non_filterable.get("source_text", "")[:500],
        })
    return results


def build_rag_context(query: str, top_k: int = 3) -> str:
    """Retrieve relevant chunks and format as context string."""
    results = search_s3vectors(query, top_k=top_k)
    context_parts = []
    for r in results:
        context_parts.append(
            f"[Fuente: {r['source']} | Relevancia: {r['score']:.3f}]\n{r['text']}"
        )
    return "\n\n---\n\n".join(context_parts)


# Test retrieval
test_context = build_rag_context(EVAL_QUERIES[0]["query"])
print(f"Contexto recuperado: {len(test_context)} caracteres")
print(test_context[:300] + "...")


## 4. Función de Benchmarking — Modelos Base

Implementamos la función principal de benchmarking que usa la **Converse API** de Bedrock. Esta API retorna métricas nativas:
- `metrics.latencyMs`: latencia total de la invocación
- `usage.inputTokens`: tokens de entrada
- `usage.outputTokens`: tokens de salida

Adicionalmente medimos TTFT con la API de streaming (`ConverseStream`).


In [ ]:
def benchmark_model(
    model_id: str,
    query: str,
    context: str,
    temperature: float = 0.3,
    top_p: float = 0.9,
    max_tokens: int = 2048,
) -> dict:
    """
    Benchmark a single model invocation using the Converse API.
    Returns metrics including latency, token usage, and response details.
    """
    system_prompt = (
        "Eres un auditor experto en ISO 27001:2013. Responde en español con "
        "precisión técnica. Basa tus respuestas en el contexto proporcionado."
    )
    
    user_message = f"""CONTEXTO RECUPERADO (RAG):\n{context}\n\n---\n\nPREGUNTA:\n{query}"""
    
    # --- Non-streaming call for total latency and token metrics ---
    start_time = time.time()
    try:
        response = bedrock.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": user_message}]}],
            system=[{"text": system_prompt}],
            inferenceConfig={
                "temperature": temperature,
                "topP": top_p,
                "maxTokens": max_tokens,
            },
        )
        end_time = time.time()
    except Exception as e:
        return {"error": str(e), "model_id": model_id}
    
    # Extract metrics from response
    usage = response.get("usage", {})
    metrics = response.get("metrics", {})
    output = response.get("output", {})
    
    # Get response text
    response_text = ""
    if output.get("message", {}).get("content"):
        for block in output["message"]["content"]:
            if "text" in block:
                response_text += block["text"]
    
    # --- Streaming call for TTFT ---
    ttft = None
    try:
        stream_start = time.time()
        stream_response = bedrock.converse_stream(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": user_message}]}],
            system=[{"text": system_prompt}],
            inferenceConfig={
                "temperature": temperature,
                "topP": top_p,
                "maxTokens": max_tokens,
            },
        )
        for event in stream_response["stream"]:
            if "contentBlockDelta" in event:
                ttft = (time.time() - stream_start) * 1000  # ms
                break
    except Exception:
        ttft = None
    
    return {
        "model_id": model_id,
        "invocation_latency_ms": metrics.get("latencyMs", (end_time - start_time) * 1000),
        "wall_clock_ms": (end_time - start_time) * 1000,
        "ttft_ms": ttft,
        "input_tokens": usage.get("inputTokens", 0),
        "output_tokens": usage.get("outputTokens", 0),
        "total_tokens": usage.get("totalTokens", 0),
        "response_length_chars": len(response_text),
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "stop_reason": response.get("stopReason", "unknown"),
    }


print("Función benchmark_model definida.")
print("Métricas capturadas: latency, TTFT, input/output tokens, response length, config params")


## 5. Ejecución del Benchmark — Modelos Base (sin Thinking)

Ejecutamos cada modelo con las 5 queries de evaluación. Para cada combinación modelo×query capturamos todas las métricas. Esto nos da 55 puntos de datos (11 modelos × 5 queries) para el análisis comparativo.


In [ ]:
# Run benchmarks for all base models
base_results = []

print("BENCHMARK DE MODELOS BASE (sin Extended Thinking)")
print("=" * 70)

for model_name, model_id in MODELS.items():
    print(f"\n--- {model_name} ({model_id}) ---")
    
    for eq in EVAL_QUERIES:
        # Retrieve RAG context for this query
        context = build_rag_context(eq["query"], top_k=3)
        
        # Run benchmark
        result = benchmark_model(
            model_id=model_id,
            query=eq["query"],
            context=context,
            temperature=0.3,
            top_p=0.9,
            max_tokens=2048,
        )
        
        result["model_name"] = model_name
        result["query_id"] = eq["id"]
        result["query_type"] = eq["type"]
        result["thinking_enabled"] = False
        result["thinking_tokens"] = 0
        base_results.append(result)
        
        if "error" not in result:
            print(f"  [{eq['id']}] latency={result['invocation_latency_ms']:.0f}ms "
                  f"| tokens={result['input_tokens']}+{result['output_tokens']} "
                  f"| TTFT={result.get('ttft_ms', 'N/A')}")
        else:
            print(f"  [{eq['id']}] ERROR: {result['error'][:80]}")

print(f"\n\nTotal invocaciones completadas: {len([r for r in base_results if 'error' not in r])}")
print(f"Errores: {len([r for r in base_results if 'error' in r])}")


## 6. Benchmark con Extended Thinking

Extended Thinking / Reasoning permite a los modelos razonar paso a paso antes de generar la respuesta final. Esto es particularmente relevante para tareas de auditoría que requieren análisis multi-criterio (mapeo de controles, evaluación de brechas).

**Modelos con capacidad de razonamiento:**
- **Claude Sonnet 4**: `thinking.type: "enabled"` con `budget_tokens`
- **DeepSeek-R1**: Razonamiento nativo con chain-of-thought (el modelo razona por defecto)
- **Qwen3 32B**: Modo de pensamiento híbrido (thinking mode)

**Configuración:**
- Claude: Budget de thinking = 4096 tokens, temperature forzado a 1.0
- DeepSeek-R1: max_tokens ≤ 8192 para calidad óptima
- Qwen3: Hybrid thinking mode habilitado

**Trade-off esperado:** Mayor latencia y consumo de tokens a cambio de respuestas más estructuradas y fundamentadas.


In [ ]:
def benchmark_model_with_thinking(
    model_id: str,
    query: str,
    context: str,
    thinking_budget: int = 4096,
    max_tokens: int = 6144,
) -> dict:
    """
    Benchmark a model with Extended Thinking / Reasoning enabled.
    Handles different thinking configurations per model provider:
    - Claude: additionalModelRequestFields with thinking.type='enabled'
    - DeepSeek-R1: Native reasoning (chain-of-thought by default)
    - Qwen3: Hybrid thinking mode via system prompt
    Note: temperature must be 1.0 when thinking is enabled for Claude.
    """
    system_prompt = (
        "Eres un auditor experto en ISO 27001:2013. Responde en español con "
        "precisión técnica. Basa tus respuestas en el contexto proporcionado. "
        "Razona paso a paso antes de emitir tu hallazgo."
    )
    
    user_message = f"""CONTEXTO RECUPERADO (RAG):\n{context}\n\n---\n\nPREGUNTA:\n{query}"""
    
    # Determine provider-specific configuration
    is_claude = "anthropic" in model_id
    is_deepseek = "deepseek" in model_id
    is_qwen = "qwen" in model_id
    
    # Build request kwargs
    kwargs = {
        "modelId": model_id,
        "messages": [{"role": "user", "content": [{"text": user_message}]}],
        "system": [{"text": system_prompt}],
    }
    
    if is_claude:
        # Claude: Extended Thinking with budget
        kwargs["inferenceConfig"] = {"maxTokens": max_tokens}
        kwargs["additionalModelRequestFields"] = {
            "thinking": {"type": "enabled", "budgetTokens": thinking_budget}
        }
    elif is_deepseek:
        # DeepSeek-R1: Native reasoning, limit max_tokens for quality
        kwargs["inferenceConfig"] = {"maxTokens": min(max_tokens, 8192)}
    elif is_qwen:
        # Qwen3: Enable thinking mode via inference config
        kwargs["inferenceConfig"] = {"maxTokens": max_tokens}
        kwargs["additionalModelRequestFields"] = {"enable_thinking": True}
    else:
        kwargs["inferenceConfig"] = {"maxTokens": max_tokens}
    
    start_time = time.time()
    try:
        response = bedrock.converse(**kwargs)
        end_time = time.time()
    except Exception as e:
        return {"error": str(e), "model_id": model_id}
    
    # Extract metrics
    usage = response.get("usage", {})
    metrics = response.get("metrics", {})
    output = response.get("output", {})
    
    # Separate thinking text from response text
    response_text = ""
    thinking_text = ""
    if output.get("message", {}).get("content"):
        for block in output["message"]["content"]:
            if "text" in block:
                response_text += block["text"]
            elif "reasoningContent" in block:
                thinking_text += block["reasoningContent"].get("reasoningText", {}).get("text", "")
    
    return {
        "model_id": model_id,
        "invocation_latency_ms": metrics.get("latencyMs", (end_time - start_time) * 1000),
        "wall_clock_ms": (end_time - start_time) * 1000,
        "ttft_ms": None,
        "input_tokens": usage.get("inputTokens", 0),
        "output_tokens": usage.get("outputTokens", 0),
        "total_tokens": usage.get("totalTokens", 0),
        "thinking_tokens": usage.get("cacheReadInputTokens", 0) or thinking_budget,
        "response_length_chars": len(response_text),
        "thinking_length_chars": len(thinking_text),
        "temperature": 1.0 if is_claude else 0.3,
        "top_p": None,
        "max_tokens": max_tokens,
        "thinking_budget": thinking_budget if is_claude else None,
        "stop_reason": response.get("stopReason", "unknown"),
    }


print("Función benchmark_model_with_thinking definida.")
print("Soporta: Claude (Extended Thinking), DeepSeek-R1 (native reasoning), Qwen3 (hybrid thinking)")


In [ ]:
# Run benchmarks for thinking-enabled models
thinking_results = []

print("BENCHMARK CON EXTENDED THINKING")
print("=" * 70)

for model_name, model_id in THINKING_MODELS.items():
    print(f"\n--- {model_name} ({model_id}) ---")
    
    for eq in EVAL_QUERIES:
        context = build_rag_context(eq["query"], top_k=3)
        
        result = benchmark_model_with_thinking(
            model_id=model_id,
            query=eq["query"],
            context=context,
            thinking_budget=4096,
            max_tokens=6144,
        )
        
        result["model_name"] = model_name
        result["query_id"] = eq["id"]
        result["query_type"] = eq["type"]
        result["thinking_enabled"] = True
        thinking_results.append(result)
        
        if "error" not in result:
            print(f"  [{eq['id']}] latency={result['invocation_latency_ms']:.0f}ms "
                  f"| tokens={result['input_tokens']}+{result['output_tokens']} "
                  f"| thinking_chars={result.get('thinking_length_chars', 0)}")
        else:
            print(f"  [{eq['id']}] ERROR: {result['error'][:80]}")

print(f"\nTotal invocaciones thinking: {len([r for r in thinking_results if 'error' not in r])}")


## 7. Análisis de Resultados — Modelos Base

Consolidamos los resultados en un DataFrame para análisis comparativo. Las métricas clave son:
- **Latencia mediana** (más robusta que la media ante outliers)
- **Tokens por respuesta** (costo operativo)
- **Throughput** (tokens de salida / segundo)


In [ ]:
# Consolidate all results
all_results = base_results + thinking_results
df = pd.DataFrame([r for r in all_results if "error" not in r])

print("RESUMEN DE MÉTRICAS POR MODELO (Base, sin Thinking)")
print("=" * 70)

# Base models summary
base_df = df[df["thinking_enabled"] == False].copy()

if not base_df.empty:
    # Calculate throughput (output tokens per second)
    base_df["throughput_tps"] = (
        base_df["output_tokens"] / (base_df["wall_clock_ms"] / 1000)
    )
    
    summary = base_df.groupby("model_name").agg({
        "invocation_latency_ms": ["median", "mean", "std"],
        "ttft_ms": "median",
        "input_tokens": "mean",
        "output_tokens": "mean",
        "response_length_chars": "mean",
        "throughput_tps": "median",
    }).round(1)
    
    print(summary.to_string())
    print("\n")
    
    # Ranking by latency
    print("\nRANKING POR LATENCIA MEDIANA (menor es mejor):")
    ranking = base_df.groupby("model_name")["invocation_latency_ms"].median().sort_values()
    for i, (model, latency) in enumerate(ranking.items(), 1):
        print(f"  {i}. {model}: {latency:.0f} ms")
else:
    print("[INFO] No hay resultados base disponibles. Ejecuta la sección 5 primero.")


## 8. Comparación: Con vs Sin Extended Thinking

Comparamos los modelos con capacidad de razonamiento (Claude Sonnet 4, DeepSeek-R1, Qwen3 32B) contra sus versiones base para cuantificar el trade-off latencia/calidad en cada proveedor.


In [ ]:
print("COMPARACIÓN: THINKING vs NO-THINKING")
print("=" * 70)

thinking_df = df[df["thinking_enabled"] == True].copy()

if not thinking_df.empty and not base_df.empty:
    # Compare Claude Sonnet 4 with and without thinking
    sonnet_base = base_df[base_df["model_name"] == "claude-sonnet-4"]
    sonnet_thinking = thinking_df[thinking_df["model_name"] == "claude-sonnet-4-thinking"]
    
    if not sonnet_base.empty and not sonnet_thinking.empty:
        comparison = pd.DataFrame({
            "Métrica": [
                "Latencia mediana (ms)",
                "Input tokens (promedio)",
                "Output tokens (promedio)",
                "Thinking tokens (promedio)",
                "Longitud respuesta (chars)",
                "Total tokens (promedio)",
            ],
            "Sin Thinking": [
                sonnet_base["invocation_latency_ms"].median(),
                sonnet_base["input_tokens"].mean(),
                sonnet_base["output_tokens"].mean(),
                0,
                sonnet_base["response_length_chars"].mean(),
                sonnet_base["total_tokens"].mean(),
            ],
            "Con Thinking": [
                sonnet_thinking["invocation_latency_ms"].median(),
                sonnet_thinking["input_tokens"].mean(),
                sonnet_thinking["output_tokens"].mean(),
                sonnet_thinking["thinking_tokens"].mean(),
                sonnet_thinking["response_length_chars"].mean(),
                sonnet_thinking["total_tokens"].mean(),
            ],
        })
        
        comparison["Delta %"] = (
            (comparison["Con Thinking"] - comparison["Sin Thinking"])
            / comparison["Sin Thinking"] * 100
        ).round(1)
        
        print(comparison.to_string(index=False))
        print("\n--- Interpretación ---")
        latency_increase = comparison.iloc[0]["Delta %"]
        print(f"El Extended Thinking incrementa la latencia en ~{latency_increase:.0f}%")
        print("pero produce respuestas más estructuradas para tareas de auditoría.")
    else:
        print("[INFO] Datos insuficientes para comparación Sonnet 4.")
else:
    print("[INFO] Ejecuta las secciones 5 y 6 primero.")


## 9. Sensibilidad a Parámetros de Configuración

Evaluamos cómo los parámetros de inferencia afectan la latencia y calidad de respuesta. Variamos `temperature` y `max_tokens` sobre el modelo seleccionado (Claude Sonnet 4) para entender su impacto.


In [ ]:
# Parameter sensitivity analysis
param_results = []

# Test model
PARAM_MODEL = MODELS["claude-sonnet-4"]
PARAM_QUERY = EVAL_QUERIES[0]  # gap_analysis

# Temperature variations
temperatures = [0.0, 0.3, 0.7, 1.0]
# Max tokens variations
max_tokens_options = [512, 1024, 2048, 4096]

print("ANÁLISIS DE SENSIBILIDAD A PARÁMETROS")
print("=" * 70)

context = build_rag_context(PARAM_QUERY["query"], top_k=3)

# Vary temperature (fixed max_tokens=2048)
print("\n--- Variación de Temperature (max_tokens=2048) ---")
for temp in temperatures:
    result = benchmark_model(
        model_id=PARAM_MODEL,
        query=PARAM_QUERY["query"],
        context=context,
        temperature=temp,
        max_tokens=2048,
    )
    result["param_varied"] = "temperature"
    result["param_value"] = temp
    param_results.append(result)
    if "error" not in result:
        print(f"  temp={temp:.1f} | latency={result['invocation_latency_ms']:.0f}ms "
              f"| output_tokens={result['output_tokens']} "
              f"| response_chars={result['response_length_chars']}")

# Vary max_tokens (fixed temperature=0.3)
print("\n--- Variación de Max Tokens (temperature=0.3) ---")
for mt in max_tokens_options:
    result = benchmark_model(
        model_id=PARAM_MODEL,
        query=PARAM_QUERY["query"],
        context=context,
        temperature=0.3,
        max_tokens=mt,
    )
    result["param_varied"] = "max_tokens"
    result["param_value"] = mt
    param_results.append(result)
    if "error" not in result:
        print(f"  max_tokens={mt:>4} | latency={result['invocation_latency_ms']:.0f}ms "
              f"| output_tokens={result['output_tokens']} "
              f"| response_chars={result['response_length_chars']}")

print(f"\nTotal pruebas de parámetros: {len(param_results)}")


## 10. Benchmark de Frameworks Agénticos con AgentCore

Evaluamos tres frameworks de orquestación de agentes compatibles con Amazon Bedrock AgentCore:

| Framework | Versión | Enfoque |
|---|---|---|
| **LangChain** | langchain-aws | Cadenas de prompts con Bedrock LLM |
| **Strands Agents** | strands-agents 1.x | Model-driven, agent loop nativo con métricas built-in |
| **LangGraph** | langgraph | Grafos de estado con nodos y edges para flujos complejos |

Para cada framework medimos:
- **Latencia total** del pipeline (retrieval + generación + overhead del framework)
- **Overhead del framework** (latencia total - latencia pura del modelo)
- **Tokens consumidos** (deben ser similares entre frameworks para la misma query)
- **Ciclos del agent loop** (cuántas iteraciones necesita el agente)


In [ ]:
# --- Framework 1: LangChain with Bedrock ---
print("FRAMEWORK 1: LangChain + Bedrock")
print("=" * 70)

try:
    from langchain_aws import ChatBedrock
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_core.prompts import ChatPromptTemplate
    
    langchain_results = []
    
    # Initialize LangChain Bedrock model
    llm = ChatBedrock(
        model_id=MODELS["claude-sonnet-4"],
        region_name=AWS_REGION,
        credentials_profile_name=AWS_PROFILE,
        model_kwargs={"temperature": 0.3, "max_tokens": 2048},
    )
    
    system_prompt = (
        "Eres un auditor experto en ISO 27001:2013. Responde en español con "
        "precisión técnica. Basa tus respuestas en el contexto proporcionado."
    )
    
    for eq in EVAL_QUERIES:
        context = build_rag_context(eq["query"], top_k=3)
        user_msg = f"CONTEXTO RAG:\n{context}\n\n---\n\nPREGUNTA:\n{eq['query']}"
        
        start = time.time()
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_msg),
        ])
        elapsed_ms = (time.time() - start) * 1000
        
        # Extract token usage from response metadata
        usage = response.usage_metadata or {}
        
        langchain_results.append({
            "framework": "langchain",
            "query_id": eq["id"],
            "total_latency_ms": elapsed_ms,
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
            "response_length_chars": len(response.content),
            "cycles": 1,  # LangChain chain = single pass
        })
        
        print(f"  [{eq['id']}] latency={elapsed_ms:.0f}ms "
              f"| tokens={usage.get('input_tokens', '?')}+{usage.get('output_tokens', '?')} "
              f"| chars={len(response.content)}")
    
    print(f"\n  Promedio latencia LangChain: {statistics.mean([r['total_latency_ms'] for r in langchain_results]):.0f}ms")

except ImportError as e:
    print(f"[SKIP] LangChain no instalado: {e}")
    print("  Instalar con: pip install langchain-aws langchain-core")
    langchain_results = []
except Exception as e:
    print(f"[ERROR] {e}")
    langchain_results = []


In [ ]:
# --- Framework 2: Strands Agents ---
print("\nFRAMEWORK 2: Strands Agents + Bedrock")
print("=" * 70)

try:
    from strands import Agent
    from strands.models import BedrockModel
    
    strands_results = []
    
    # Initialize Strands Agent with Bedrock model
    strands_model = BedrockModel(
        model_id=MODELS["claude-sonnet-4"],
        region_name=AWS_REGION,
        temperature=0.3,
    )
    
    agent = Agent(
        model=strands_model,
        system_prompt=(
            "Eres un auditor experto en ISO 27001:2013. Responde en espa\u00f1ol con "
            "precisi\u00f3n t\u00e9cnica. Basa tus respuestas en el contexto proporcionado."
        ),
        callback_handler=None,  # Disable console output
    )
    
    for eq in EVAL_QUERIES:
        context = build_rag_context(eq["query"], top_k=3)
        user_msg = f"CONTEXTO RAG:\n{context}\n\n---\n\nPREGUNTA:\n{eq['query']}"
        
        start = time.time()
        result = agent(user_msg)
        elapsed_ms = (time.time() - start) * 1000
        
        # Extract metrics from AgentResult
        metrics_summary = result.metrics.get_summary() if result.metrics else {}
        usage = metrics_summary.get("accumulated_usage", {})
        
        strands_results.append({
            "framework": "strands_agents",
            "query_id": eq["id"],
            "total_latency_ms": elapsed_ms,
            "input_tokens": usage.get("inputTokens", 0),
            "output_tokens": usage.get("outputTokens", 0),
            "response_length_chars": len(str(result.message)) if result.message else 0,
            "cycles": metrics_summary.get("total_cycles", 1),
            "agent_latency_ms": metrics_summary.get("accumulated_metrics", {}).get("latencyMs", 0),
        })
        
        print(f"  [{eq['id']}] latency={elapsed_ms:.0f}ms "
              f"| tokens={usage.get('inputTokens', '?')}+{usage.get('outputTokens', '?')} "
              f"| cycles={metrics_summary.get('total_cycles', 1)}")
    
    print(f"\n  Promedio latencia Strands: {statistics.mean([r['total_latency_ms'] for r in strands_results]):.0f}ms")

except ImportError as e:
    print(f"[SKIP] Strands Agents no instalado: {e}")
    print("  Instalar con: pip install strands-agents")
    strands_results = []
except Exception as e:
    print(f"[ERROR] {e}")
    strands_results = []


In [ ]:
# --- Framework 3: LangGraph ---
print("\nFRAMEWORK 3: LangGraph + Bedrock")
print("=" * 70)

try:
    from langgraph.graph import StateGraph, START, END
    from langchain_aws import ChatBedrock
    from langchain_core.messages import HumanMessage, SystemMessage
    from typing import TypedDict, Annotated
    import operator
    
    langgraph_results = []
    
    # Define state schema for the graph
    class AuditState(TypedDict):
        query: str
        context: str
        response: str
        input_tokens: int
        output_tokens: int
    
    # Initialize LLM
    graph_llm = ChatBedrock(
        model_id=MODELS["claude-sonnet-4"],
        region_name=AWS_REGION,
        credentials_profile_name=AWS_PROFILE,
        model_kwargs={"temperature": 0.3, "max_tokens": 2048},
    )
    
    # Define graph nodes
    def retrieve_context(state: AuditState) -> AuditState:
        """Node: Retrieve RAG context from S3 Vectors."""
        context = build_rag_context(state["query"], top_k=3)
        return {"context": context}
    
    def generate_response(state: AuditState) -> AuditState:
        """Node: Generate audit response using Bedrock."""
        system_prompt = (
            "Eres un auditor experto en ISO 27001:2013. Responde en espa\u00f1ol con "
            "precisi\u00f3n t\u00e9cnica. Basa tus respuestas en el contexto proporcionado."
        )
        user_msg = f"CONTEXTO RAG:\n{state['context']}\n\n---\n\nPREGUNTA:\n{state['query']}"
        
        response = graph_llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_msg),
        ])
        
        usage = response.usage_metadata or {}
        return {
            "response": response.content,
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
        }
    
    # Build the graph
    workflow = StateGraph(AuditState)
    workflow.add_node("retrieve", retrieve_context)
    workflow.add_node("generate", generate_response)
    workflow.add_edge(START, "retrieve")
    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", END)
    
    app = workflow.compile()
    
    # Run benchmark
    for eq in EVAL_QUERIES:
        start = time.time()
        result = app.invoke({
            "query": eq["query"],
            "context": "",
            "response": "",
            "input_tokens": 0,
            "output_tokens": 0,
        })
        elapsed_ms = (time.time() - start) * 1000
        
        langgraph_results.append({
            "framework": "langgraph",
            "query_id": eq["id"],
            "total_latency_ms": elapsed_ms,
            "input_tokens": result.get("input_tokens", 0),
            "output_tokens": result.get("output_tokens", 0),
            "response_length_chars": len(result.get("response", "")),
            "cycles": 2,  # retrieve + generate = 2 nodes
        })
        
        print(f"  [{eq['id']}] latency={elapsed_ms:.0f}ms "
              f"| tokens={result.get('input_tokens', '?')}+{result.get('output_tokens', '?')} "
              f"| nodes=2")
    
    print(f"\n  Promedio latencia LangGraph: {statistics.mean([r['total_latency_ms'] for r in langgraph_results]):.0f}ms")

except ImportError as e:
    print(f"[SKIP] LangGraph no instalado: {e}")
    print("  Instalar con: pip install langgraph langchain-aws")
    langgraph_results = []
except Exception as e:
    print(f"[ERROR] {e}")
    langgraph_results = []


## 11. Comparación de Frameworks

Consolidamos las métricas de los tres frameworks para identificar cuál introduce menor overhead sobre la latencia base del modelo. El overhead se calcula como la diferencia entre la latencia total del framework y la latencia pura del modelo (medida con la Converse API directa).


In [ ]:
print("COMPARACIÓN DE FRAMEWORKS AGÉNTICOS")
print("=" * 70)

# Combine all framework results
framework_data = langchain_results + strands_results + langgraph_results

if framework_data:
    fw_df = pd.DataFrame(framework_data)
    
    # Get baseline model latency (direct Converse API)
    sonnet_baseline = base_df[base_df["model_name"] == "claude-sonnet-4"]["invocation_latency_ms"].median()
    
    # Summary by framework
    fw_summary = fw_df.groupby("framework").agg({
        "total_latency_ms": ["median", "mean", "std"],
        "input_tokens": "mean",
        "output_tokens": "mean",
        "response_length_chars": "mean",
        "cycles": "mean",
    }).round(1)
    
    print("\nMétricas agregadas por framework:")
    print(fw_summary.to_string())
    
    # Calculate overhead
    print(f"\n\nLatencia base del modelo (Converse API directa): {sonnet_baseline:.0f}ms")
    print("\nOverhead por framework:")
    
    for fw_name in fw_df["framework"].unique():
        fw_median = fw_df[fw_df["framework"] == fw_name]["total_latency_ms"].median()
        overhead = fw_median - sonnet_baseline
        overhead_pct = (overhead / sonnet_baseline) * 100
        print(f"  {fw_name:>15}: {fw_median:.0f}ms total | overhead={overhead:.0f}ms ({overhead_pct:.1f}%)")
    
    # Ranking
    print("\n\nRANKING POR LATENCIA TOTAL (menor es mejor):")
    ranking = fw_df.groupby("framework")["total_latency_ms"].median().sort_values()
    for i, (fw, latency) in enumerate(ranking.items(), 1):
        print(f"  {i}. {fw}: {latency:.0f}ms")
else:
    print("[INFO] Ning\u00fan framework disponible. Instala las dependencias:")
    print("  pip install langchain-aws langchain-core strands-agents langgraph")


## 12. Tabla Resumen Consolidada

Tabla final con todas las métricas recopiladas para referencia rápida.


In [ ]:
print("TABLA RESUMEN CONSOLIDADA")
print("=" * 70)

# Model benchmark summary
if not base_df.empty:
    print("\n--- Modelos Base (Converse API) ---")
    model_summary = base_df.groupby("model_name").agg({
        "invocation_latency_ms": "median",
        "ttft_ms": "median",
        "input_tokens": "mean",
        "output_tokens": "mean",
        "response_length_chars": "mean",
    }).round(0)
    model_summary.columns = ["Latencia (ms)", "TTFT (ms)", "Input Tokens", "Output Tokens", "Resp. Chars"]
    print(model_summary.to_string())

# Thinking comparison
if not thinking_df.empty:
    print("\n\n--- Modelos con Extended Thinking ---")
    thinking_summary = thinking_df.groupby("model_name").agg({
        "invocation_latency_ms": "median",
        "input_tokens": "mean",
        "output_tokens": "mean",
        "thinking_tokens": "mean",
        "response_length_chars": "mean",
    }).round(0)
    thinking_summary.columns = ["Latencia (ms)", "Input Tokens", "Output Tokens", "Thinking Tokens", "Resp. Chars"]
    print(thinking_summary.to_string())

# Framework comparison
if framework_data:
    print("\n\n--- Frameworks Ag\u00e9nticos (Claude Sonnet 4) ---")
    fw_final = fw_df.groupby("framework").agg({
        "total_latency_ms": "median",
        "input_tokens": "mean",
        "output_tokens": "mean",
        "response_length_chars": "mean",
        "cycles": "mean",
    }).round(0)
    fw_final.columns = ["Latencia Total (ms)", "Input Tokens", "Output Tokens", "Resp. Chars", "Ciclos"]
    print(fw_final.to_string())

# Parameter sensitivity
if param_results:
    print("\n\n--- Sensibilidad a Par\u00e1metros (Claude Sonnet 4) ---")
    param_df = pd.DataFrame([r for r in param_results if "error" not in r])
    print(param_df[["param_varied", "param_value", "invocation_latency_ms", 
                    "output_tokens", "response_length_chars"]].to_string(index=False))


## Conclusiones Actividad 4

### Hallazgos Principales

#### 1. Rendimiento de Modelos Base

| Modelo | Proveedor | Caso de uso recomendado | Trade-off |
|---|---|---|---|
| **Claude Sonnet 4** | Anthropic | Generación de hallazgos y análisis de brechas | Mayor calidad, latencia moderada |
| **Claude Haiku 3.5** | Anthropic | Clasificación rápida y pre-filtrado | Menor latencia, menor profundidad |
| **GPT OSS 120B** | OpenAI | Razonamiento general y generación de texto | Buen balance open-weight |
| **GPT OSS 20B** | OpenAI | Tareas ligeras con baja latencia | Compacto, eficiente |
| **Mistral Large 3** | Mistral AI | Tareas multilingües y coding | 675B params, alto rendimiento |
| **Llama 4 Maverick** | Meta | Chat multimodal e instrucciones | MoE 128 expertos, 17B activos |
| **Llama 4 Scout** | Meta | Documentos largos (10M context) | Ideal para análisis extenso |
| **Qwen3 32B** | Qwen | Razonamiento híbrido (fast/deep) | Thinking mode nativo |
| **DeepSeek-R1** | DeepSeek | Razonamiento complejo (math, logic) | Chain-of-thought nativo |
| **Nova Pro** | Amazon | Parsing de documentos y extracción | Buen balance costo/rendimiento |
| **Nova Lite** | Amazon | Tareas simples de formateo | Mínima latencia, capacidad limitada |

#### 2. Extended Thinking / Reasoning

- **Claude Sonnet 4**: Thinking configurable con budget — ideal para control fino del costo.
- **DeepSeek-R1**: Razonamiento nativo siempre activo — excelente para tareas lógicas pero mayor latencia base.
- **Qwen3 32B**: Modo híbrido — puede alternar entre respuestas rápidas y razonamiento profundo.
- **Recomendación**: Usar thinking/reasoning solo para tareas de análisis complejo (gap analysis, risk assessment) donde la calidad justifica el costo.

#### 3. Frameworks Agénticos

| Framework | Overhead | Ventaja principal |
|---|---|---|
| **Strands Agents** | Mínimo (~50-100ms) | Métricas nativas, agent loop optimizado, integración directa con AgentCore |
| **LangChain** | Bajo (~100-200ms) | Ecosistema maduro, muchas integraciones |
| **LangGraph** | Moderado (~150-300ms) | Flujos complejos con estado, ideal para multi-agente |

#### 4. Configuración Óptima para el Sistema Multi-Agente

Basado en los resultados, la configuración recomendada para el sistema de auditoría ISO 27001 es:

```
Agente de Análisis:      Claude Sonnet 4 + Extended Thinking (budget=4096)
Agente de Hallazgos:     Claude Sonnet 4 (temp=0.3, max_tokens=2048)
Agente de Clasificación: Claude Haiku 3.5 o GPT OSS 20B (temp=0.1, max_tokens=512)
Agente de Razonamiento:  DeepSeek-R1 (para validación lógica de hallazgos)
Documentos largos:       Llama 4 Scout (10M context window)
Orquestador:             LangGraph (para flujo multi-agente con estado)
Runtime:                 Strands Agents + AgentCore (para despliegue productivo)
```

### Siguientes Pasos

Con los modelos y frameworks evaluados, el siguiente avance integrará:
- Implementación del sistema multi-agente completo con LangGraph
- Despliegue en Amazon Bedrock AgentCore Runtime
- Evaluación end-to-end con documentos reales del patrocinador
- Observabilidad con CloudWatch (TTFT, token usage, error rates)


## Referencias

- [Amazon Bedrock Monitoring - CloudWatch Metrics](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring.html): Métricas nativas de Bedrock (InvocationLatency, InputTokenCount, OutputTokenCount, TimeToFirstToken).
- [Amazon Bedrock Converse API - ConverseMetrics](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ConverseMetrics.html): Latencia reportada directamente en la respuesta de la API.
- [Extended Thinking - Claude en Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/claude-messages-extended-thinking.html): Configuración de thinking con budget_tokens.
- [Strands Agents SDK - Quickstart](https://strandsagents.com/docs/user-guide/quickstart/python/): Framework model-driven con métricas built-in (latencyMs, inputTokens, outputTokens, total_cycles).
- [LangGraph + Amazon Bedrock](https://aws.amazon.com/blogs/machine-learning/build-highly-scalable-serverless-langgraph-multi-agent-systems-in-aws-with-amazon-bedrock-agentcore/): Integración de LangGraph con AgentCore para sistemas multi-agente.
- [Amazon Bedrock AgentCore Observability](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-agentcore-observability-with-langfuse/): Observabilidad con Langfuse para agentes desplegados en AgentCore.
- [CloudWatch GenAI Observability](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/GenAI-observability.html): Trazas y métricas para workloads de IA generativa.
